## Initial Transformation and Data Cleaning

In [1]:
# https://developer.nvidia.com/blog/7-drop-in-replacements-to-instantly-speed-up-your-python-data-science-workflows/

# Just add this to the top of your script!
#%load_ext cudf.pandas

import pandas as pd

# Importing custom functions 
import useful_functions as uf

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Load customer data(using relative path)
customers_df = pd.read_csv('data/customers.csv', delimiter='|')
uf.show_df_info(customers_df)


First few rows of the DataFrame:
           ssn               cc_num    first       last gender  \
0  115-04-4507        4218196001337    Kathy    Johnson      F   
1  715-55-5575  4351161559407816183   Elaine     Fuller      F   
2  167-48-5821        4192832764832  Melinda    Cameron      F   
3  406-83-7518     4238849696532874  Brandon   Williams      M   
4  697-93-1877     4514627048281480     Lisa  Hernandez      F   

                       street           city state    zip      lat      long  \
0        863 Lawrence Valleys  Staten Island    NY  10302  40.6306  -74.1379   
1  310 Kendra Common Apt. 164        Peabody    MA   1960  42.5326  -70.9612   
2            05641 Robin Port       Waukomis    OK  73773  36.2781  -97.8996   
3      26916 Carlson Mountain    Los Angeles    CA  90019  34.0482 -118.3343   
4             809 Burns Creek         Austin    TX  78727  30.4254  -97.7195   

   city_pop                                    job         dob      acct_num  \
0    468

In [3]:
# Splitting the profile column into multiple columns
# https://pandas.pydata.org/docs/reference/api/pandas.Series.str.rsplit.html

profile_parts = customers_df['profile'].str.rsplit('_', n=2, expand=True)
profile_parts.head()

,0,1,2
0,adults_2550,female,urban.json
1,adults_2550,female,urban.json
2,adults_50up,female,rural.json
3,adults_2550,male,urban.json
4,adults_2550,female,urban.json


In [4]:
# Leveraging the information from the profile column to create new columns(Feature Engineering)

customers_df['pop_group'] = profile_parts[0]
customers_df['location'] = profile_parts[2].str.split('.').str[0] # Remove .json extension

# Drop the original profile column
customers_df.drop(columns=['profile'], inplace=True)

del profile_parts # Free memory

customers_df.head()

,ssn,cc_num,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,acct_num,pop_group,location
0,115-04-4507,4218196001337,Kathy,Johnson,F,863 Lawrence Valleys,Staten Island,NY,10302,40.6306,-74.1379,468730,Accounting technician,1982-10-03,888022315787,adults_2550,urban
1,715-55-5575,4351161559407816183,Elaine,Fuller,F,310 Kendra Common Apt. 164,Peabody,MA,1960,42.5326,-70.9612,50944,Professor Emeritus,1994-06-07,917558277935,adults_2550,urban
2,167-48-5821,4192832764832,Melinda,Cameron,F,05641 Robin Port,Waukomis,OK,73773,36.2781,-97.8996,1744,International aid/development worker,1934-05-30,718172762479,adults_50up,rural
3,406-83-7518,4238849696532874,Brandon,Williams,M,26916 Carlson Mountain,Los Angeles,CA,90019,34.0482,-118.3343,2383912,Seismic interpreter,1991-12-26,947268892251,adults_2550,urban
4,697-93-1877,4514627048281480,Lisa,Hernandez,F,809 Burns Creek,Austin,TX,78727,30.4254,-97.7195,940359,Medical laboratory scientific officer,1998-05-22,888335239225,adults_2550,urban


####  Planning to use as less memory as possible, some of the data types will be optimized


#### Introducing this code after discussing the project. Will play with customized functions.

https://stackoverflow.com/questions/57856010/automatically-optimizing-pandas-dtypes

In [5]:
# Playing with the AI functions to optimize the dataframe memory usage(will use mines anyway)
df_dict_types = uf.analyze_dataframe_for_optimization(customers_df)
before_df = customers_df.copy()
after_df = uf.optimize_df_types(customers_df, df_dict_types)

uf.print_optimization_report(before_df, after_df, df_dict_types)

# Memory cleanup
import gc

del before_df, after_df, df_dict_types
gc.collect(); # Force garbage collection, otherwise memory will not be released immediately
# The ; at the end avoids printing the output of the last command

DataFrame Memory Usage Optimization Report

---- Per-column memory usage and dtype ----
           Before (bytes)  After (bytes)  Delta (bytes) Before dtype  \
Index                 128            128              0          NaN   
Total              787231         531371         255860                
acct_num             8080           8080              0        int64   
cc_num               8080           8080              0        int64   
city                66587          66587              0       object   
city_pop             8080           4040           4040        int64   
dob                 67670          67670              0       object   
first               63639          63639              0       object   
gender              58580           1234          57346       object   
job                 78671          78671              0       object   
last                63774          63774              0       object   
lat                  8080           4040        

#### End of the test with AI generated funcions

In [6]:
# Display unique values in columns that might be used as categorical
customers_df[['gender', 'state', 'job', 'pop_group','location']].nunique()

gender         2
state         50
job          492
pop_group      3
location       2
dtype: int64

In [7]:
# Check what type city_pop should use
city_pop_recommended_type = uf.get_safe_int_type(customers_df['city_pop'])
print(f"Recommended type for city_pop: {city_pop_recommended_type}")

Recommended type for city_pop: uint32


In [8]:
# Before optimization
print("Before optimization:")
old_mem_usage = customers_df.memory_usage(deep=True).sum() / 1024
customers_df.info(memory_usage='deep')

# Apply optimization
new_customer_types = {
    'category': ['gender', 'state', 'pop_group', 'location'],
    'string': ['ssn', 'cc_num', 'first', 'last', 'street', 'city', 'dob', 'job', 'zip', 'acct_num'],
    city_pop_recommended_type: ['city_pop'],
    'float32': ['lat', 'long'] # Acceptable loss for fraud detection. It might need to be float64 for other applications
}
# Replacing customers_df with the optimized version to save memory
customers_df = uf.optimize_df_types(customers_df, new_customer_types)

# After optimization  

new_mem_usage = customers_df.memory_usage(deep=True).sum() / 1024
print("\nAfter optimization:")
customers_df.info(memory_usage='deep')

Before optimization:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ssn        1010 non-null   object 
 1   cc_num     1010 non-null   int64  
 2   first      1010 non-null   object 
 3   last       1010 non-null   object 
 4   gender     1010 non-null   object 
 5   street     1010 non-null   object 
 6   city       1010 non-null   object 
 7   state      1010 non-null   object 
 8   zip        1010 non-null   int64  
 9   lat        1010 non-null   float64
 10  long       1010 non-null   float64
 11  city_pop   1010 non-null   int64  
 12  job        1010 non-null   object 
 13  dob        1010 non-null   object 
 14  acct_num   1010 non-null   int64  
 15  pop_group  1010 non-null   object 
 16  location   1010 non-null   object 
dtypes: float64(2), int64(4), object(11)
memory usage: 768.8 KB

After optimization:
<class 'pandas.core.frame.D

In [9]:
# Compare memory usage

print(f"Original memory usage: {old_mem_usage:,.2f} KB, optimized memory usage: {new_mem_usage:,.2f} KB")

print(f"\nMemory usage reduction: {100*(old_mem_usage - new_mem_usage)/old_mem_usage:.2f}%")

Original memory usage: 768.78 KB, optimized memory usage: 699.50 KB

Memory usage reduction: 9.01%


##### Not a huge saving, but every bit helps! This principle might be useful when working with larger datasets.

#### As expected, the majority of the customers are from urban areas. Adults over 50 are the majority on our small customer dataset.

### We will work now with the transaction files

In [10]:
# The transaction data is split across multiple files, so we need to load them all and concatenate them into a single DataFrame
transaction_files = uf.get_files_dir('data', file_mask='*.csv')
transaction_files = [f for f in transaction_files if 'customers.csv' not in f] # Exclude customers.csv

In [11]:
sample_transaction_file = transaction_files[0]
sample_transactions_df = pd.read_csv(sample_transaction_file, delimiter='|')
uf.show_df_info(sample_transactions_df)


First few rows of the DataFrame:
           ssn                         trans_num  trans_date trans_time  \
0  894-92-8883  ad93731421f1158b37ed3b491eba1c5b  2020-12-17   00:50:00   
1  894-92-8883  074fa7bf648ff3cdd9bc82df487997c0  2020-12-17   03:50:30   
2  894-92-8883  72228caa4bfac623dbbeae4784fc2e01  2020-12-17   00:39:48   
3  894-92-8883  ab22aef766917bd0baf8fb631edd731d  2020-12-17   00:12:28   
4  894-92-8883  8c12c7cc4e32b3c5effea606beb99caa  2020-12-18   23:20:26   

    unix_time       category     amt  is_fraud  \
0  1608184200  gas_transport    8.61         1   
1  1608195030    grocery_pos   11.67         1   
2  1608183588    grocery_pos  304.66         1   
3  1608181948  gas_transport  335.67         1   
4  1608351626   shopping_net  677.02         1   

                          merchant  merch_lat  merch_long  
0                  fraud_Kling Inc  44.010097  -88.728769  
1  fraud_Heller, Gutmann and Zieme  42.172962  -87.049088  
2                fraud_Lockman Ltd

#### Unix time
##### https://en.wikipedia.org/wiki/Unix_time

In [12]:
sample_transactions_df.dtypes

ssn            object
trans_num      object
trans_date     object
trans_time     object
unix_time       int64
category       object
amt           float64
is_fraud        int64
merchant       object
merch_lat     float64
merch_long    float64
dtype: object

In [13]:
# Working with the transaction data 
# Optimizing types
transaction_types = {
    'category': ['category'],
    'uint32': ['unix_time'], # uint32 max value is 2,147,483,647, which is more than enough for our purposes(until 2038, at least). 
    'float32': ['amt', 'merch_lat', 'merch_long'],
    'uint8': ['is_fraud'],
    'string': ['ssn', 'trans_num', 'trans_date', 'trans_time', 'merchant']
}
# Process and merge all transaction files
transactions_df = uf.process_and_merge_files(transaction_files, transaction_types)

Processing file 1/53: data/adults_50up_male_urban_0808-1009.csv
Processed 178,533 rows so far...
Processing file 2/53: data/adults_2550_male_urban_0404-0605.csv
Processing file 3/53: data/adults_50up_male_rural_0606-0807.csv
Processing file 4/53: data/adults_50up_female_rural_0808-1009.csv
Processing file 5/53: data/adults_2550_male_rural_0404-0605.csv
Processing file 6/53: data/young_adults_male_urban_0404-0605.csv
Processing file 7/53: data/adults_50up_male_urban_0202-0403.csv
Processing file 8/53: data/young_adults_male_urban_0000-0201.csv
Processing file 9/53: data/adults_2550_female_rural_0000-0201.csv
Processing file 10/53: data/young_adults_female_rural_0808-1009.csv
Processing file 11/53: data/adults_2550_female_rural_0404-0605.csv
Processed 876,238 rows so far...
Processing file 12/53: data/adults_2550_female_urban_0000-0201.csv
Processing file 13/53: data/adults_50up_female_urban_0202-0403.csv
Processing file 14/53: data/adults_2550_male_rural_0202-0403.csv
Processing file 15

In [14]:
# Display the final merged DataFrame
uf.show_df_info(transactions_df)


First few rows of the DataFrame:
           ssn                         trans_num  trans_date trans_time  \
0  894-92-8883  ad93731421f1158b37ed3b491eba1c5b  2020-12-17   00:50:00   
1  894-92-8883  074fa7bf648ff3cdd9bc82df487997c0  2020-12-17   03:50:30   
2  894-92-8883  72228caa4bfac623dbbeae4784fc2e01  2020-12-17   00:39:48   
3  894-92-8883  ab22aef766917bd0baf8fb631edd731d  2020-12-17   00:12:28   
4  894-92-8883  8c12c7cc4e32b3c5effea606beb99caa  2020-12-18   23:20:26   

    unix_time       category         amt  is_fraud  \
0  1608184200  gas_transport    8.610000         1   
1  1608195030    grocery_pos   11.670000         1   
2  1608183588    grocery_pos  304.660004         1   
3  1608181948  gas_transport  335.670013         1   
4  1608351626   shopping_net  677.020020         1   

                          merchant  merch_lat  merch_long  
0                  fraud_Kling Inc  44.010098  -88.728767  
1  fraud_Heller, Gutmann and Zieme  42.172962  -87.049088  
2         

In [15]:
help(uf.get_files_dir)

Help on function get_files_dir in module useful_functions:

get_files_dir(directory_path: str, file_mask: str = '*.csv') -> list
    Get all files matching the pattern in a directory.
    
    Args:
        directory_path: Path to the directory containing files
        file_mask: File pattern to match (default: '*.csv')
    
    Returns:
        list: List of file paths matching the pattern



In [16]:
# 1. Validate primary key candidates
customer_pk_check = uf.check_primary_key_candidates(customers_df, ['ssn', 'cc_num'])
transaction_pk_check = uf.check_primary_key_candidates(transactions_df, ['trans_num'])

In [17]:
print("Customer Primary Key Analysis:")
uf.display_primary_key_analysis( customer_pk_check)
print("\nTransaction Primary Key Analysis:")
uf.display_primary_key_analysis( transaction_pk_check)

Customer Primary Key Analysis:

Primary Key Analysis for 'ssn':
  Total rows: 1,010
  Unique values: 1,010
  Null values: 0
  Duplicate values: 0
  Status: VALID_PRIMARY_KEY

Primary Key Analysis for 'cc_num':
  Total rows: 1,010
  Unique values: 1,010
  Null values: 0
  Duplicate values: 0
  Status: VALID_PRIMARY_KEY

Transaction Primary Key Analysis:

Primary Key Analysis for 'trans_num':
  Total rows: 4,740,009
  Unique values: 4,740,009
  Null values: 0
  Duplicate values: 0
  Status: VALID_PRIMARY_KEY


In [ ]:
# Create database
# Remove the comment to create the database
# uf.create_fraud_detection_db(customers_df, transactions_df)

Database created successfully with proper primary keys, foreign keys, and indexes


In [ ]:
# Memory cleanup
import gc

# Show current memory usage
customers_memory_mb = customers_df.memory_usage(deep=True).sum() / (1024 * 1024)
transactions_memory_mb = transactions_df.memory_usage(deep=True).sum() / (1024 * 1024)
sample_memory_mb = sample_transactions_df.memory_usage(deep=True).sum() / (1024 * 1024)
total_memory_mb = customers_memory_mb + transactions_memory_mb + sample_memory_mb

print(f"Current memory usage:")
print(f"Customers: {customers_memory_mb:.2f} MB")
print(f"Transactions: {transactions_memory_mb:.2f} MB")
print(f"Sample: {sample_memory_mb:.2f} MB")
print(f"Total: {total_memory_mb:.2f} MB")

# Clean up variables
del customers_df, transactions_df, sample_transactions_df
del transaction_files, sample_transaction_file
del new_customer_types, transaction_types
del customer_pk_check, transaction_pk_check
del old_mem_usage, new_mem_usage, city_pop_recommended_type

gc.collect()
print(f"\nFreed {total_memory_mb:.2f} MB of memory")

Current memory usage:
Customers: 0.68 MB
Transactions: 1749.91 MB
Sample: 81.13 MB
Total: 1831.71 MB


NameError: name 'profile_parts' is not defined